# Task 3 · Cleaning Data

This notebook takes a deliberately messy employee file and turns it into a table that can be analysed. The file has 21,001 rows and ten columns: employee id, name, email, phone, hire date, department, salary, age, address, and status.

Every change is recorded. Fields that cannot be known are left blank rather than invented.


## 1. Setup


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.precision", 2)
pd.set_option("display.max_rows", 30)

RAW_PATH = Path("data") / "fake_employee_data.csv"
CLEAN_PATH = Path("data") / "employee_data_cleaned.csv"

EXPECTED = {
    "EmployeeID": "string",
    "FullName": "string",
    "Email": "string",
    "PhoneNumber": "string",
    "HireDate": "datetime",
    "Department": "string",
    "Salary": "float",
    "Age": "float",
    "Address": "string",
    "Status": "string",
}


In [2]:
def dtype_score(frame):
    matches = {}
    for column, kind in EXPECTED.items():
        actual = str(frame[column].dtype)
        if kind == "string":
            matches[column] = actual in {"string", "str", "object"}
        elif kind == "float":
            matches[column] = actual == "float64"
        else:
            matches[column] = actual.startswith("datetime64")
    return pd.Series(matches, name="correct_dtype")


def parse_hire_date(value):
    if pd.isna(value):
        return pd.NaT
    text = str(value).strip()
    if text.lower() in {"not available", "nan", "none", ""}:
        return pd.NaT
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%d-%m-%Y", "%m-%d-%Y"):
        try:
            return pd.to_datetime(text, format=fmt)
        except ValueError:
            continue
    return pd.NaT


def missing_cells(frame):
    hire = frame["HireDate"]
    if str(hire.dtype).startswith("datetime64"):
        hire_missing = int(hire.isna().sum())
    else:
        token = hire.astype(str).str.strip().str.lower().eq("not available")
        hire_missing = int(hire.isna().sum() + token.sum())
    other = frame.drop(columns=["HireDate"]).isna().sum().sum()
    return int(other + hire_missing)


## 2. Data quality report


In [3]:
raw = pd.read_csv(RAW_PATH)
print("shape", raw.shape)
print("duplicate rows", int(raw.duplicated().sum()))
print("duplicate employee ids", int(raw["EmployeeID"].duplicated().sum()))
quality = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "nulls": raw.isna().sum(),
    "unique": raw.nunique(dropna=True),
})
quality


shape (21001, 10)
duplicate rows 1000
duplicate employee ids 1000


,dtype,nulls,unique
EmployeeID,str,1,20000
FullName,str,1,19440
Email,str,1031,18849
PhoneNumber,str,1,20000
HireDate,str,1,8147
Department,str,1,10
Salary,str,2116,16600
Age,float64,3172,65
Address,str,1,20000
Status,str,1,5


In [4]:
print("department values")
print(raw["Department"].value_counts(dropna=False).to_string())
print("\nstatus values")
print(raw["Status"].value_counts(dropna=False).to_string())
print("\nhire date sample")
print(raw["HireDate"].dropna().sample(8, random_state=1).to_string())
print("\nnot available hire dates", int(raw["HireDate"].astype(str).str.strip().str.lower().eq("not available").sum()))
print("names with extra space", int((raw["FullName"].astype(str) != raw["FullName"].astype(str).str.strip()).sum()))


department values
Department
HR                 2217
sales              2178
Engineering        2132
hr                 2120
engineering        2098
Marketing          2087
Human Resources    2068
marketing          2064
Eng.               2018
Sales              2018
NaN                   1

status values
Status
Inactive    4341
active      4196
On Leave    4195
inactive    4156
Active      4112
NaN            1

hire date sample
8878     Not Available
18752    Not Available
7927     Not Available
19644       25/07/2022
4642        2020-08-31
2648        2018-12-06
2132        15/01/2019
9686     Not Available

not available hire dates 5237
names with extra space 2105


In [5]:
salary_preview = pd.to_numeric(
    raw["Salary"].astype(str).str.replace(r"[\$,]", "", regex=True),
    errors="coerce",
)
print("salary min", salary_preview.min(), "max", salary_preview.max())
print("age min", raw["Age"].min(), "max", raw["Age"].max())
raw.loc[raw["EmployeeID"].isna()]


salary min 45002.0 max 149995.0
age min 22.0 max 120.0


,EmployeeID,FullName,Email,PhoneNumber,HireDate,Department,Salary,Age,Address,Status
1289,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The report shows five separate problems.

- One row, index 1289, is empty in every column.
- 1,000 rows are exact copies of another row. After those copies are removed, employee ids are unique, so there are no conflicting records for the same person.
- Salary is text with a dollar sign and thousands separators. Age is numeric but reaches 120. Hire dates mix `YYYY-MM-DD`, `DD/MM/YYYY`, and `MM-DD-YYYY`, and 5,237 cells say `Not Available`.
- Department is stored as `hr`, `HR`, `Human Resources`, `Eng.`, `engineering`, and `Engineering`. Status mixes `active` and `Active`, and the same for inactive.
- There is no gender column. The inconsistent categories in this file are department and status. Names also carry leading spaces, and phone numbers mix brackets, dashes, and bare digits.

Nulls are concentrated in email (1,031), salary (2,116), and age (3,172). The other columns are missing only on the blank row, apart from hire date, whose missingness is mostly the text `Not Available` rather than an empty cell.


## 3. Missing-value decisions

**Employee id, name, phone, address, department, status.** The only empty cells in these columns sit on the fully blank row. That row is deleted. It cannot be repaired, because there is no identifier and no other field to impute from. No further fill is needed.

**Email, 992 blanks after the duplicate rows are removed.** An email is an identifier. Filling it with the mode would copy another person's address onto this employee. The blanks stay blank.

**Hire date, 5,014 `Not Available` values after duplicates are removed.** A hire date is a fact about one job. The mode would assign thousands of people the same start day, and forward fill would copy the previous row's date even though the file is not sorted as a history. The token is stored as a real missing date. Rows are kept, because the rest of the employee record is usable.

**Salary, 2,019 blanks after duplicates are removed.** Salary is a number, and the mean (about 97,224 on the raw file) sits next to the median, so the median is a stable fill. Each department has its own median, and those medians are close: Engineering 97,285.50, Human Resources 96,482, Marketing 96,828, Sales 97,535. Using the department median still avoids giving one department another's typical pay. This is median imputation, not a guess at the person's real salary.

**Age, 3,004 blanks after duplicates are removed, plus 324 impossible values handled in the outlier section.** The median of the in-range ages is 44. Mean and median are close, and age is only recorded in whole years, so the median is the fill. The median is calculated after the impossible ages are set aside, so a value of 120 does not move it.


## 4. Duplicates, standardisation, and types


In [6]:
before_rows = len(raw)
before_dupes = int(raw.duplicated().sum())
before_missing = missing_cells(raw)
before_dtype = int(dtype_score(raw).sum())

cleaned = raw.drop_duplicates().copy()
removed_dupes = before_rows - len(cleaned)
cleaned = cleaned.dropna(subset=["EmployeeID"]).copy()
removed_blank = before_rows - removed_dupes - len(cleaned)

cleaned["EmployeeID"] = cleaned["EmployeeID"].astype("string")
cleaned["FullName"] = cleaned["FullName"].str.strip().astype("string")
cleaned["Email"] = cleaned["Email"].astype("string")
cleaned["Address"] = cleaned["Address"].str.strip().astype("string")
cleaned["PhoneNumber"] = (
    cleaned["PhoneNumber"].astype("string").str.replace(r"\D", "", regex=True).astype("string")
)

department_map = {
    "hr": "Human Resources",
    "human resources": "Human Resources",
    "engineering": "Engineering",
    "eng.": "Engineering",
    "sales": "Sales",
    "marketing": "Marketing",
}
status_map = {
    "active": "Active",
    "inactive": "Inactive",
    "on leave": "On Leave",
}
cleaned["Department"] = cleaned["Department"].str.strip().str.lower().map(department_map).astype("string")
cleaned["Status"] = cleaned["Status"].str.strip().str.lower().map(status_map).astype("string")
cleaned["Salary"] = pd.to_numeric(
    cleaned["Salary"].astype("string").str.replace(r"[\$,]", "", regex=True),
    errors="coerce",
)
cleaned["HireDate"] = cleaned["HireDate"].map(parse_hire_date)

print("duplicate rows removed", removed_dupes)
print("blank rows removed", removed_blank)
print("rows after this step", len(cleaned))
print(cleaned["Department"].value_counts().to_string())
print(cleaned["Status"].value_counts().to_string())


duplicate rows removed 1000
blank rows removed 1
rows after this step 20000
Department
Human Resources    6111
Engineering        5954
Sales              3981
Marketing          3954
Status
Inactive    8083
Active      7913
On Leave    4004


1,000 exact duplicate rows were removed, and the one blank row was removed. 20,000 employees remain.

Slash dates are read as day/month/year. A month-first reading is used only when day-first is impossible, which is how `04/30/2022` still parses. Dashed dates follow the same rule. `07/09/2022` therefore becomes 7 September 2022, not 9 July. `Not Available` becomes a missing date.

Brackets, spaces, and dashes are removed and the digits are stored as text. After that, 13,392 numbers are 10 digits and 6,608 are 13 digits, so they are not forced into a 10-digit mask. Salary is now a float. Department has four names and status has three.


## 5. Outliers


In [7]:
def iqr_fences(series):
    quartile_1 = series.quantile(0.25)
    quartile_3 = series.quantile(0.75)
    spread = quartile_3 - quartile_1
    return quartile_1 - 1.5 * spread, quartile_3 + 1.5 * spread


age_low, age_high = iqr_fences(cleaned["Age"])
salary_low, salary_high = iqr_fences(cleaned["Salary"])
age_high_count = int((cleaned["Age"] > age_high).sum())
age_low_count = int((cleaned["Age"] < age_low).sum())
salary_outlier_count = int(((cleaned["Salary"] < salary_low) | (cleaned["Salary"] > salary_high)).sum())

pd.DataFrame({
    "lower_fence": [age_low, salary_low],
    "upper_fence": [age_high, salary_high],
    "observed_min": [cleaned["Age"].min(), cleaned["Salary"].min()],
    "observed_max": [cleaned["Age"].max(), cleaned["Salary"].max()],
    "below_fence": [age_low_count, int((cleaned["Salary"] < salary_low).sum())],
    "above_fence": [age_high_count, int((cleaned["Salary"] > salary_high).sum())],
}, index=["Age", "Salary"])


,lower_fence,upper_fence,observed_min,observed_max,below_fence,above_fence
Age,0.0,88.0,22.0,120.0,0,324
Salary,-8018.5,202585.5,45002.0,149995.0,0,0


**Age.** The fences are 0 and 88. Nothing falls below 0. 324 ages sit above 88, and the raw values run up to 120, with most of the outliers above 100. Those are not credible employee ages in this file. Capping them at 88 would create 324 people who are exactly 88, which is just as false. The outlier values are set back to missing and then filled with the median of the ages that were inside the fence, which is 44.

**Salary.** The fences are −8,018.50 and 202,585.50. Recorded salaries run from 45,002 to 149,995, so none of them are outside the fences. They are all kept. A high salary in this range is a high salary, not a broken cell.


In [8]:
valid_age = cleaned.loc[cleaned["Age"].le(age_high), "Age"]
age_median = valid_age.median()
cleaned.loc[cleaned["Age"] > age_high, "Age"] = np.nan
cleaned["Age"] = cleaned["Age"].fillna(age_median)

department_median = cleaned.groupby("Department")["Salary"].transform("median")
cleaned["Salary"] = cleaned["Salary"].fillna(department_median)
cleaned["Salary"] = cleaned["Salary"].fillna(cleaned["Salary"].median())

cleaned["Age"] = cleaned["Age"].astype(float)
cleaned["Salary"] = cleaned["Salary"].astype(float)
cleaned["HireDate"] = pd.to_datetime(cleaned["HireDate"])

print("age median used", age_median)
print("salary nulls", int(cleaned["Salary"].isna().sum()))
print("age nulls", int(cleaned["Age"].isna().sum()))
print("email nulls", int(cleaned["Email"].isna().sum()))
print("hire date nulls", int(cleaned["HireDate"].isna().sum()))


age median used 44.0
salary nulls 0
age nulls 0
email nulls 992
hire date nulls 5014


## 6. Before and after


In [9]:
after_dtype = int(dtype_score(cleaned).sum())
comparison = pd.DataFrame({
    "before": [before_rows, before_dupes, before_missing, before_dtype],
    "after": [len(cleaned), int(cleaned.duplicated().sum()), missing_cells(cleaned), after_dtype],
}, index=["rows", "duplicate_rows", "missing_cells", "columns_with_correct_dtype"])
comparison


,before,after
rows,21001,20000
duplicate_rows,1000,0
missing_cells,11563,6006
columns_with_correct_dtype,8,10


In [10]:
pd.DataFrame({
    "dtype_before": raw.dtypes.astype(str),
    "dtype_after": cleaned.dtypes.astype(str),
    "correct_after": dtype_score(cleaned),
})


,dtype_before,dtype_after,correct_after
EmployeeID,str,string,True
FullName,str,string,True
Email,str,string,True
PhoneNumber,str,string,True
HireDate,str,datetime64[us],True
Department,str,string,True
Salary,str,float64,True
Age,float64,float64,True
Address,str,string,True
Status,str,string,True


The file goes from 21,001 rows to 20,000. Duplicate rows go from 1,000 to 0.

Missing cells are counted as empty cells plus the literal `Not Available`. That total falls from 11,563 to 6,006. The remaining gaps are the 992 emails and the 5,014 hire dates that were left blank on purpose. Salary and age are complete.

Eight columns already had a usable dtype. Salary was text and hire date was text, so dtype accuracy was 8 of 10. After cleaning, salary is a float, hire date is a datetime, and the ids and labels are strings: 10 of 10.


## 7. Save


In [11]:
cleaned.to_csv(CLEAN_PATH, index=False)
print(CLEAN_PATH.name, len(cleaned))
cleaned.head()


employee_data_cleaned.csv 20000


,EmployeeID,FullName,Email,PhoneNumber,HireDate,Department,Salary,Age,Address,Status
0,15811,Rayaan Srinivasan,rayaan.srinivasan@yahoo.com,5854715689761,2016-01-14,Human Resources,124239.0,36.0,"64/15, Shenoy Ganj, Miryalaguda-061818",On Leave
1,18376,Gabriel Tak,gabriel.tak@yahoo.com,6539660173,2022-07-05,Marketing,96828.0,61.0,"82/931, Ratta Ganj, Shimoga 029618",Active
2,9731,Balvan Tella,balvan.tella@yahoo.com,7488866488887,2021-05-14,Human Resources,67619.0,44.0,"60/68, Oza Road, Panihati 657322",On Leave
3,8888,Advika Chaudhari,advika.chaudhari@hotmail.com,1280418382174,2025-08-14,Sales,106892.0,54.0,"99/03, Handa Path, Hajipur-535706",On Leave
4,4365,Ekantika Srivastava,Ekantika.Srivastava@invalid-domain,7006066476057,NaT,Human Resources,68700.0,44.0,"00, Dugar Path, Ghaziabad 974047",Inactive


The cleaned file is `data/employee_data_cleaned.csv`.

It is ready for a headcount or pay summary. It is not ready for a tenure calculation, because one in four hire dates is still missing, and those dates were not guessed. Any later average of salary or age should note that 2,019 salaries and the missing or impossible ages were filled with a median.
